# Notebook 1: Descriptor Extraction (Multi-Dataset)
Extracts **E_pfrac** (p-orbital fraction) descriptors from `vasprun.xml` for Rashba, Dresselhaus, and Unmatched compounds.

**Window:** 1.0 eV (proven best results)

**Descriptor Type:**
- **E** `p_frac`: total p-orbital fraction at VBM/CBM

**Output CSVs:** 
- `desc_E_p1.0_rashba.csv`
- `desc_E_p1.0_dresselhaus.csv`
- `desc_E_p1.0_unmatched.csv`


In [37]:
import numpy as np
import pandas as pd
import os
import glob
import warnings
from pathlib import Path
from pymatgen.io.vasp.outputs import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType
from pymatgen.core.periodic_table import Element

warnings.filterwarnings('ignore')

In [38]:
# ============================================================
# PATHS - MULTI-DATASET CONFIGURATION
# ============================================================
BASE_DIR = r"C:\Users\AbCMS_Lab\Desktop\Keshav-DDP"
OUTPUT_DIR = os.path.join(BASE_DIR, "Weight-contribution", "contribution-model")
INVERSE_DESIGN_DIR = os.path.join(BASE_DIR, "Inverse-design")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define datasets to process
# For rashba & dresselhaus: (name, csv_path, folder_path)
# For unmatched: (name, None, folder_path) -- no CSV, scan folder directly
DATASETS = [
    ('rashba', os.path.join(BASE_DIR, "Data", "rashba.csv"), os.path.join(INVERSE_DESIGN_DIR, "rashba")),
    ('dresselhaus', os.path.join(BASE_DIR, "Data", "dresselhaus.csv"), os.path.join(INVERSE_DESIGN_DIR, "dresselhaus")),
    ('unmatched', None, os.path.join(INVERSE_DESIGN_DIR, "unmatched")),  # No CSV, scan folder
]

# Only compute E_pfrac at window 1.0 eV
WINDOWS = [1.0]

print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Datasets to process:")
for name, csv_path, folder_path in DATASETS:
    csv_status = "✓ CSV" if csv_path and os.path.exists(csv_path) else ("✗ CSV" if csv_path else "-- (no CSV)")
    folder_exists = "✓" if os.path.isdir(folder_path) else "✗"
    print(f"  {name:15} | {csv_status:10} | Folder: {folder_exists} {folder_path}")

Base directory: C:\Users\AbCMS_Lab\Desktop\Keshav-DDP
Output directory: C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Weight-contribution\contribution-model
Datasets to process:
  rashba          | ✓ CSV      | Folder: ✓ C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\rashba
  dresselhaus     | ✓ CSV      | Folder: ✓ C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\dresselhaus
  unmatched       | -- (no CSV) | Folder: ✓ C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\unmatched


In [39]:
# ============================================================
# LOAD CSV & GET TARGETS (or scan folder for unmatched)
# ============================================================
def load_target_data(csv_path, folder_path, dataset_name):
    """Load CSV or scan folder directly for unmatched dataset."""
    
    # Case 1: No CSV provided (unmatched) - scan folder for vasprun files
    if csv_path is None:
        print(f"\n{dataset_name.upper()}")
        print(f"  No CSV provided. Scanning folder for vasprun.xml files...")
        
        target_list = []
        if not os.path.isdir(folder_path):
            print(f"  ⚠ Folder not found: {folder_path}")
            return None
        
        for folder in os.listdir(folder_path):
            folder_path_full = os.path.join(folder_path, folder)
            if not os.path.isdir(folder_path_full):
                continue
            
            # Try named pattern first: ss_2d%2F{folder}%2Fbands_ncl%2Fvasprun.xml
            vasprun_name = f"ss_2d%2F{folder}%2Fbands_ncl%2Fvasprun.xml"
            vasprun_path = os.path.join(folder_path_full, vasprun_name)
            
            # Fallback: glob search
            if not os.path.exists(vasprun_path):
                candidates = glob.glob(os.path.join(folder_path_full, "**", "vasprun.xml"), recursive=True)
                if not candidates:
                    continue
                vasprun_path = candidates[0]
            
            # Extract UID and Formula from folder name
            # Format: Formula-uid (e.g., AsBrTe-671e6de2497a)
            last_dash = folder.rfind('-')
            if last_dash >= 0:
                formula = folder[:last_dash]
                uid = folder[last_dash+1:]
            else:
                formula = folder
                uid = folder
            
            target_list.append({
                'uid': uid,
                'Formula': formula,
                'alpha_R_max': 0  # placeholder for unmatched
            })
        
        target = pd.DataFrame(target_list)
        print(f"  Found UIDs: {len(target)}")
        print(f"  Target compounds: {len(target)}")
        return target if len(target) > 0 else None
    
    # Case 2: CSV provided (rashba, dresselhaus)
    if not os.path.exists(csv_path):
        print(f"⚠ {dataset_name.upper()}: CSV not found at {csv_path}")
        return None
    
    df = pd.read_csv(csv_path)
    print(f"\n{dataset_name.upper()}")
    print(f"  CSV rows: {len(df)}")
    print(f"  Unique UIDs: {df['uid'].nunique()}")
    print(f"  Columns: {list(df.columns)}")
    
    # Get max Rashba_parameter per UID if available; otherwise use 0
    target_col = 'Rashba_parameter'
    if target_col not in df.columns:
        print(f"  ⚠ '{target_col}' not in columns. Using 0 as placeholder.")
        target = df.groupby('uid').first().reset_index()[['uid']]
        target['alpha_R_max'] = 0
    else:
        target = df.groupby('uid')[target_col].max().reset_index()
        target.columns = ['uid', 'alpha_R_max']
    
    # Keep formula for reference
    uid_formula = df.groupby('uid')['Formula'].first().reset_index()
    target = target.merge(uid_formula, on='uid')
    
    print(f"  Target compounds: {len(target)}")
    return target

# Load all targets
targets = {}
for dataset_name, csv_path, folder_path in DATASETS:
    target_data = load_target_data(csv_path, folder_path, dataset_name)
    if target_data is not None:
        targets[dataset_name] = target_data
    else:
        print(f"  SKIPPING {dataset_name}")

print(f"\nLoaded {len(targets)} datasets for processing")


RASHBA
  CSV rows: 205
  Unique UIDs: 99
  Columns: ['Formula', 'uid', 'spacegroup', 'ehull', 'bandgap', 'band', 'kpath', 'Rashba_parameter', 'SS', 'dE', 'anticrossing']
  Target compounds: 99

DRESSELHAUS
  CSV rows: 62
  Unique UIDs: 25
  Columns: ['Formula', 'uid', 'spacegroup', 'ehull', 'bandgap', 'band', 'kpath', 'Rashba_parameter', 'SS', 'dE', 'anticrossing']
  Target compounds: 25

UNMATCHED
  No CSV provided. Scanning folder for vasprun.xml files...
  Found UIDs: 78
  Target compounds: 78

Loaded 3 datasets for processing


In [40]:
# ============================================================
# DEBUG: Check unmatched folder structure
# ============================================================
unmatched_folder = os.path.join(INVERSE_DESIGN_DIR, "unmatched")
print(f"\nDEBUG: Checking {unmatched_folder}")
print(f"  Exists: {os.path.isdir(unmatched_folder)}")

if os.path.isdir(unmatched_folder):
    folders = [f for f in os.listdir(unmatched_folder) if os.path.isdir(os.path.join(unmatched_folder, f))]
    print(f"  Total folders: {len(folders)}")
    print(f"  First 5 folders: {folders[:5]}")
    
    # Check if ANY vasprun files exist
    all_vasprun = glob.glob(os.path.join(unmatched_folder, "**", "vasprun.xml"), recursive=True)
    print(f"  vasprun.xml files found: {len(all_vasprun)}")
    if all_vasprun:
        print(f"  First vasprun: {all_vasprun[0]}")
    
    # Also check the named pattern
    named_pattern = glob.glob(os.path.join(unmatched_folder, "**", "ss_2d*", "vasprun.xml"), recursive=True)
    print(f"  vasprun files with ss_2d pattern: {len(named_pattern)}")



DEBUG: Checking C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Inverse-design\unmatched
  Exists: True
  Total folders: 78
  First 5 folders: ['Ag2Se2-248fffac5959', 'Ag2Te2-648de0a225fe', 'Al2Br6-fab8ea0c0979', 'Al2Cl6-450a3fc3ff50', 'Al2I6-29ae3f5ef5a7']
  vasprun.xml files found: 0
  vasprun files with ss_2d pattern: 0


In [41]:
# ============================================================
# FIND VASPRUN FILES FOR EACH DATASET
# ============================================================
def find_vasprun_files(folder_path, target_df):
    """Find vasprun.xml files matching target UIDs."""
    compound_dirs = {}
    unmatched_uids = []
    
    if not os.path.isdir(folder_path):
        print(f"  ⚠ Folder not found: {folder_path}")
        return compound_dirs, unmatched_uids
    
    # Scan folder for compound directories
    for folder in os.listdir(folder_path):
        folder_path_full = os.path.join(folder_path, folder)
        if not os.path.isdir(folder_path_full):
            continue
        
        # Try to find vasprun.xml
        # First try: ss_2d%2F{folder}%2Fbands_ncl%2Fvasprun.xml
        vasprun_name = f"ss_2d%2F{folder}%2Fbands_ncl%2Fvasprun.xml"
        vasprun_path = os.path.join(folder_path_full, vasprun_name)
        
        # Fallback: glob search
        if not os.path.exists(vasprun_path):
            candidates = glob.glob(os.path.join(folder_path_full, "**", "vasprun.xml"), recursive=True)
            if candidates:
                vasprun_path = candidates[0]
            else:
                continue
        
        # Extract UID from folder name
        last_dash = folder.rfind('-')
        if last_dash >= 0:
            uid_from_folder = folder[last_dash+1:]
        else:
            uid_from_folder = folder
        
        compound_dirs[uid_from_folder] = {
            'folder': folder,
            'vasprun': vasprun_path
        }
    
    # Match with target UIDs
    matched = 0
    for _, row in target_df.iterrows():
        uid = row['uid']
        if uid in compound_dirs:
            matched += 1
        else:
            # Try partial match
            found = False
            for folder_uid in list(compound_dirs.keys()):
                if uid.startswith(folder_uid) or folder_uid.startswith(uid):
                    compound_dirs[uid] = compound_dirs[folder_uid]
                    matched += 1
                    found = True
                    break
            if not found:
                unmatched_uids.append(uid)
    
    print(f"  Found {len(compound_dirs)} compound directories")
    print(f"  Matched: {matched}/{len(target_df)}")
    if unmatched_uids:
        print(f"  Unmatched: {len(unmatched_uids)} UIDs")
    
    return compound_dirs, unmatched_uids

# Find vasprun files for each dataset
compound_dirs_all = {}
for dataset_name in targets.keys():
    dataset_folder = next(f for n, _, f in DATASETS if n == dataset_name)
    print(f"\nFinding vasprun files for {dataset_name}...")
    compound_dirs, unmatched = find_vasprun_files(dataset_folder, targets[dataset_name])
    compound_dirs_all[dataset_name] = compound_dirs


Finding vasprun files for rashba...
  Found 99 compound directories
  Matched: 99/99

Finding vasprun files for dresselhaus...
  Found 25 compound directories
  Matched: 25/25

Finding vasprun files for unmatched...
  Found 78 compound directories
  Matched: 78/78


In [42]:
# ============================================================
# CORE FUNCTIONS
# ============================================================

def get_dos_array(dos_obj):
    """Extract DOS values, summing spin channels for SOC."""
    if Spin.down in dos_obj.densities:
        return dos_obj.densities[Spin.up] + dos_obj.densities[Spin.down]
    return dos_obj.densities[Spin.up]

def integrate_window(energies, dos_vals, window):
    """Integrate DOS in energy window using trapezoidal rule."""
    mask = (energies >= window[0]) & (energies <= window[1])
    e_w = energies[mask]
    d_w = dos_vals[mask]
    if len(e_w) < 2:
        return 0.0
    return np.trapezoid(d_w, e_w)

def extract_contributions(vasprun_path, window_size):
    """
    Parse vasprun.xml and extract orbital/atomic contributions at VBM/CBM.
    Returns dict with per-element per-orbital contributions (normalized).
    """
    vr = Vasprun(vasprun_path, parse_dos=True, parse_eigen=False)
    cdos = vr.complete_dos
    e_fermi = vr.efermi
    
    vbm = cdos.get_cbm_vbm()[1]
    cbm = cdos.get_cbm_vbm()[0]
    
    vbm_s = vbm - e_fermi
    cbm_s = cbm - e_fermi
    energies = cdos.energies - e_fermi
    
    vbm_win = [vbm_s - window_size, vbm_s]
    cbm_win = [cbm_s, cbm_s + window_size]
    
    element_dos = cdos.get_element_dos()
    orbital_map = {'s': OrbitalType.s, 'p': OrbitalType.p, 'd': OrbitalType.d}
    
    # Raw contributions
    raw = {}
    for element in element_dos:
        el = str(element)
        mass = Element(el).atomic_mass
        Z = Element(el).Z
        raw[el] = {'mass': float(mass), 'Z': Z}
        
        spd = cdos.get_element_spd_dos(element)
        for orb_str in ['s', 'p', 'd']:
            orb_type = orbital_map[orb_str]
            if orb_type in spd:
                dos_vals = get_dos_array(spd[orb_type])
                raw[el][f'{orb_str}_VBM'] = integrate_window(energies, dos_vals, vbm_win)
                raw[el][f'{orb_str}_CBM'] = integrate_window(energies, dos_vals, cbm_win)
            else:
                raw[el][f'{orb_str}_VBM'] = 0.0
                raw[el][f'{orb_str}_CBM'] = 0.0
    
    # Total per band edge
    total_vbm = sum(raw[el][f'{o}_VBM'] for el in raw for o in ['s','p','d'])
    total_cbm = sum(raw[el][f'{o}_CBM'] for el in raw for o in ['s','p','d'])
    
    # Normalized
    for el in raw:
        for o in ['s','p','d']:
            raw[el][f'{o}_VBM_norm'] = raw[el][f'{o}_VBM'] / total_vbm if total_vbm > 0 else 0
            raw[el][f'{o}_CBM_norm'] = raw[el][f'{o}_CBM'] / total_cbm if total_cbm > 0 else 0
        # Element-level (sum of orbitals)
        raw[el]['w_VBM'] = sum(raw[el][f'{o}_VBM_norm'] for o in ['s','p','d'])
        raw[el]['w_CBM'] = sum(raw[el][f'{o}_CBM_norm'] for o in ['s','p','d'])
        # p-orbital fraction for this element
        raw[el]['p_VBM'] = raw[el]['p_VBM_norm']
        raw[el]['p_CBM'] = raw[el]['p_CBM_norm']
    
    return raw

def compute_descriptors(raw):
    """
    From raw contributions dict, compute all descriptor types.
    Returns flat dict of descriptors.
    """
    desc = {}
    elements = list(raw.keys())
    
    # Sort by mass (heaviest first)
    sorted_els = sorted(elements, key=lambda x: raw[x]['mass'], reverse=True)
    heavy1 = sorted_els[0] if len(sorted_els) >= 1 else None
    heavy2 = sorted_els[1] if len(sorted_els) >= 2 else None
    
    # --- Type A: WM_total = sum(w_X * M_X) ---
    desc['A_WM_VBM'] = sum(raw[el]['w_VBM'] * raw[el]['mass'] for el in elements)
    desc['A_WM_CBM'] = sum(raw[el]['w_CBM'] * raw[el]['mass'] for el in elements)
    
    # --- Type B: WM_p_only = sum(p_X * M_X) ---
    desc['B_WMp_VBM'] = sum(raw[el]['p_VBM'] * raw[el]['mass'] for el in elements)
    desc['B_WMp_CBM'] = sum(raw[el]['p_CBM'] * raw[el]['mass'] for el in elements)
    
    # --- Type C: WM_p_frac = p_i*M_i / sum(p_j*M_j) for top 2 heaviest ---
    denom_vbm = sum(raw[el]['p_VBM'] * raw[el]['mass'] for el in elements)
    denom_cbm = sum(raw[el]['p_CBM'] * raw[el]['mass'] for el in elements)
    
    if heavy1:
        desc['C_pfrac_h1_VBM'] = (raw[heavy1]['p_VBM'] * raw[heavy1]['mass']) / denom_vbm if denom_vbm > 0 else 0
        desc['C_pfrac_h1_CBM'] = (raw[heavy1]['p_CBM'] * raw[heavy1]['mass']) / denom_cbm if denom_cbm > 0 else 0
    else:
        desc['C_pfrac_h1_VBM'] = 0
        desc['C_pfrac_h1_CBM'] = 0
    
    if heavy2:
        desc['C_pfrac_h2_VBM'] = (raw[heavy2]['p_VBM'] * raw[heavy2]['mass']) / denom_vbm if denom_vbm > 0 else 0
        desc['C_pfrac_h2_CBM'] = (raw[heavy2]['p_CBM'] * raw[heavy2]['mass']) / denom_cbm if denom_cbm > 0 else 0
    else:
        desc['C_pfrac_h2_VBM'] = 0
        desc['C_pfrac_h2_CBM'] = 0
    
    # --- Type D: WM_indiv = w_heavyN * M_heavyN ---
    if heavy1:
        desc['D_wm_h1_VBM'] = raw[heavy1]['w_VBM'] * raw[heavy1]['mass']
        desc['D_wm_h1_CBM'] = raw[heavy1]['w_CBM'] * raw[heavy1]['mass']
    else:
        desc['D_wm_h1_VBM'] = 0
        desc['D_wm_h1_CBM'] = 0
    
    if heavy2:
        desc['D_wm_h2_VBM'] = raw[heavy2]['w_VBM'] * raw[heavy2]['mass']
        desc['D_wm_h2_CBM'] = raw[heavy2]['w_CBM'] * raw[heavy2]['mass']
    else:
        desc['D_wm_h2_VBM'] = 0
        desc['D_wm_h2_CBM'] = 0
    
    # --- Type E: p_frac = total p-orbital % ---
    desc['E_pfrac_VBM'] = sum(raw[el]['p_VBM'] for el in elements)
    desc['E_pfrac_CBM'] = sum(raw[el]['p_CBM'] for el in elements)
    
    # --- Type F: p_heavy = p_frac_of_heaviest * M_heaviest ---
    if heavy1:
        desc['F_ph1_VBM'] = raw[heavy1]['p_VBM'] * raw[heavy1]['mass']
        desc['F_ph1_CBM'] = raw[heavy1]['p_CBM'] * raw[heavy1]['mass']
    else:
        desc['F_ph1_VBM'] = 0
        desc['F_ph1_CBM'] = 0
    
    if heavy2:
        desc['F_ph2_VBM'] = raw[heavy2]['p_VBM'] * raw[heavy2]['mass']
        desc['F_ph2_CBM'] = raw[heavy2]['p_CBM'] * raw[heavy2]['mass']
    else:
        desc['F_ph2_VBM'] = 0
        desc['F_ph2_CBM'] = 0
    
    # --- Metadata ---
    desc['heavy1_el'] = heavy1 if heavy1 else 'NA'
    desc['heavy2_el'] = heavy2 if heavy2 else 'NA'
    desc['heavy1_mass'] = raw[heavy1]['mass'] if heavy1 else 0
    desc['heavy2_mass'] = raw[heavy2]['mass'] if heavy2 else 0
    desc['n_elements'] = len(elements)
    
    return desc

In [43]:
# ============================================================
# BATCH PROCESS ALL COMPOUNDS (SIMPLIFIED: E_pfrac only)
# ============================================================
def process_dataset(dataset_name, target_df, compound_dirs):
    """Process one dataset and return results."""
    results = {w: [] for w in WINDOWS}
    failed = []
    
    print(f"\nProcessing {dataset_name.upper()}...")
    
    for idx, row in target_df.iterrows():
        uid = row['uid']
        formula = row['Formula']
        alpha_R = row['alpha_R_max']
        
        if uid not in compound_dirs:
            failed.append({'uid': uid, 'formula': formula, 'reason': 'no_folder'})
            continue
        
        vasprun_path = compound_dirs[uid]['vasprun']
        
        print(f"  [{idx+1}/{len(target_df)}] {formula} ({uid[:12]}...)", end=" ")
        
        try:
            for w in WINDOWS:
                raw = extract_contributions(vasprun_path, w)
                
                # Extract ONLY E_pfrac descriptors
                desc = {
                    'uid': uid,
                    'Formula': formula,
                    'alpha_R': alpha_R,
                    'window': w,
                    'E_pfrac_VBM': raw['E_pfrac_VBM'] if 'E_pfrac_VBM' in raw else sum(raw[el]['p_VBM'] for el in raw),
                    'E_pfrac_CBM': raw['E_pfrac_CBM'] if 'E_pfrac_CBM' in raw else sum(raw[el]['p_CBM'] for el in raw),
                }
                
                results[w].append(desc)
            print("OK")
        except Exception as e:
            failed.append({'uid': uid, 'formula': formula, 'reason': str(e)[:80]})
            print(f"FAILED: {str(e)[:60]}")
    
    print(f"  Processed: {len(results[WINDOWS[0]])} compounds")
    print(f"  Failed: {len(failed)}")
    if failed:
        for f in failed[:5]:
            print(f"    {f['formula']} ({f['uid'][:12]}): {f['reason']}")
    
    return results, failed

# Process each dataset
all_results = {}
all_failed = {}
for dataset_name in targets.keys():
    results, failed = process_dataset(
        dataset_name, 
        targets[dataset_name], 
        compound_dirs_all[dataset_name]
    )
    all_results[dataset_name] = results
    all_failed[dataset_name] = failed


Processing RASHBA...
  [1/99] SSeW (001e03f2c095...) OK
  [2/99] Sn2Te2 (03bcf7dcdaf2...) OK
  [3/99] ClSbTe (04fdd7d1ec5c...) OK
  [4/99] WMo3Se8 (05a06afa3b20...) OK
  [5/99] CrW3Se8 (0b7696e1f4c9...) OK
  [6/99] ClSbSe (0c0fbdaf8f4a...) OK
  [7/99] ISbTe (0f02957b17cf...) OK
  [8/99] AsITe (114b3382699c...) OK
  [9/99] BiBrSe (11db0908d9ef...) OK
  [10/99] CrMo3Te8 (159f028a85d0...) OK
  [11/99] TiHf3Te8 (1667d1443160...) OK
  [12/99] Ti2Zr2Te8 (18e377cce57f...) OK
  [13/99] BrSbTe (18e62ba75259...) OK
  [14/99] HgTe (1a3bdd1b142a...) OK
  [15/99] AsClSe (1a3be826b3e0...) OK
  [16/99] AsBrS (1dcd471c2288...) OK
  [17/99] O2Pb2 (20f098bd3f31...) OK
  [18/99] GeSe (211bcb7f05d6...) OK
  [19/99] MoW3Se8 (24d6cc0a0fed...) OK
  [20/99] Bi2P2S6 (287dcf4f1a19...) OK
  [21/99] BiITe (2d41b3dd1772...) OK
  [22/99] MoSTe (2ea941c8bc3c...) OK
  [23/99] MoW3S8 (2f6f133abcc8...) OK
  [24/99] BiBrTe (304bc6a92d82...) OK
  [25/99] WMo3Te8 (323fb700d903...) OK
  [26/99] ISbSe (343d2125478e...) OK


In [44]:
# ============================================================
# SAVE CSVs (E_pfrac only, dataset-specific names)
# ============================================================
saved_csvs = {}

for dataset_name in targets.keys():
    saved_csvs[dataset_name] = []
    
    for w in WINDOWS:
        if not all_results[dataset_name][w]:
            print(f"⚠ No results for {dataset_name} at window {w}")
            continue
        
        df = pd.DataFrame(all_results[dataset_name][w])
        
        # Format window as string (1.0 -> p1.0)
        w_str = f"p{w}"
        
        # CSV name format: desc_E_p1.0_{dataset}.csv
        csv_name = f"desc_E_{w_str}_{dataset_name}.csv"
        csv_path = os.path.join(OUTPUT_DIR, csv_name)
        
        df.to_csv(csv_path, index=False)
        saved_csvs[dataset_name].append(csv_name)
        
        print(f"\n{dataset_name.upper()} (window {w} eV):")
        print(f"  Saved: {csv_name}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  E_pfrac_VBM: mean={df['E_pfrac_VBM'].mean():.4f}, std={df['E_pfrac_VBM'].std():.4f}")
        print(f"  E_pfrac_CBM: mean={df['E_pfrac_CBM'].mean():.4f}, std={df['E_pfrac_CBM'].std():.4f}")

print(f"\n{'='*60}")
print(f"SUMMARY: Saved {sum(len(v) for v in saved_csvs.values())} CSV files")
for dataset_name, files in saved_csvs.items():
    for fname in files:
        fpath = os.path.join(OUTPUT_DIR, fname)
        size = os.path.getsize(fpath) if os.path.exists(fpath) else 0
        print(f"  ✓ {fname} ({size/1024:.1f} KB)")


RASHBA (window 1.0 eV):
  Saved: desc_E_p1.0_rashba.csv
  Shape: (99, 6)
  Columns: ['uid', 'Formula', 'alpha_R', 'window', 'E_pfrac_VBM', 'E_pfrac_CBM']
  E_pfrac_VBM: mean=0.6447, std=0.3082
  E_pfrac_CBM: mean=0.5503, std=0.3613

DRESSELHAUS (window 1.0 eV):
  Saved: desc_E_p1.0_dresselhaus.csv
  Shape: (25, 6)
  Columns: ['uid', 'Formula', 'alpha_R', 'window', 'E_pfrac_VBM', 'E_pfrac_CBM']
  E_pfrac_VBM: mean=0.7197, std=0.2795
  E_pfrac_CBM: mean=0.4448, std=0.3468

UNMATCHED (window 1.0 eV):
  Saved: desc_E_p1.0_unmatched.csv
  Shape: (77, 6)
  Columns: ['uid', 'Formula', 'alpha_R', 'window', 'E_pfrac_VBM', 'E_pfrac_CBM']
  E_pfrac_VBM: mean=0.7822, std=0.2532
  E_pfrac_CBM: mean=0.4987, std=0.2794

SUMMARY: Saved 3 CSV files
  ✓ desc_E_p1.0_rashba.csv (6.8 KB)
  ✓ desc_E_p1.0_dresselhaus.csv (1.7 KB)
  ✓ desc_E_p1.0_unmatched.csv (4.9 KB)


In [45]:
# ============================================================
# SANITY CHECK: INSPECT OUTPUTS
# ============================================================
print("SANITY CHECK: Loading and inspecting saved CSVs\n")

for dataset_name in targets.keys():
    w_str = "p1.0"
    csv_name = f"desc_E_{w_str}_{dataset_name}.csv"
    csv_path = os.path.join(OUTPUT_DIR, csv_name)
    
    if not os.path.exists(csv_path):
        print(f"⚠ {csv_name} not found")
        continue
    
    df = pd.read_csv(csv_path)
    print(f"{dataset_name.upper()}: {csv_name}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"\n  First 3 rows:")
    print(df.head(3).to_string(index=False))
    print()
    
    # Summary stats
    print(f"  Statistics:")
    print(f"    alpha_R range: {df['alpha_R'].min():.3f} to {df['alpha_R'].max():.3f}")
    print(f"    E_pfrac_VBM: {df['E_pfrac_VBM'].min():.3f} to {df['E_pfrac_VBM'].max():.3f}")
    print(f"    E_pfrac_CBM: {df['E_pfrac_CBM'].min():.3f} to {df['E_pfrac_CBM'].max():.3f}")
    print()

SANITY CHECK: Loading and inspecting saved CSVs

RASHBA: desc_E_p1.0_rashba.csv
  Shape: (99, 6)
  Columns: ['uid', 'Formula', 'alpha_R', 'window', 'E_pfrac_VBM', 'E_pfrac_CBM']

  First 3 rows:
         uid Formula  alpha_R  window  E_pfrac_VBM  E_pfrac_CBM
001e03f2c095    SSeW    3.288     1.0     0.224009     0.194420
03bcf7dcdaf2  Sn2Te2    4.804     1.0     0.678554     0.835895
04fdd7d1ec5c  ClSbTe    1.018     1.0     0.884091     0.920193

  Statistics:
    alpha_R range: 0.393 to 4.804
    E_pfrac_VBM: 0.134 to 0.947
    E_pfrac_CBM: 0.093 to 0.942

DRESSELHAUS: desc_E_p1.0_dresselhaus.csv
  Shape: (25, 6)
  Columns: ['uid', 'Formula', 'alpha_R', 'window', 'E_pfrac_VBM', 'E_pfrac_CBM']

  First 3 rows:
         uid Formula  alpha_R  window  E_pfrac_VBM  E_pfrac_CBM
001dfe9a7fa2   ZrSe2    1.998     1.0     0.884882     0.082029
0155c4de2320   SnBr2    1.316     1.0     0.719395     0.890736
06ebe3806790   Ir2O2    4.665     1.0     0.156249     0.308826

  Statistics:
    alph

In [46]:
# ============================================================
# FINAL SUMMARY & VERIFICATION
# ============================================================
print("\n" + "="*70)
print("PROCESSING COMPLETE - SUMMARY")
print("="*70)

summary_data = []
for dataset_name in targets.keys():
    n_target = len(targets[dataset_name])
    n_found = len(compound_dirs_all[dataset_name])
    n_processed = len(all_results[dataset_name].get(WINDOWS[0], []))
    n_failed = len(all_failed[dataset_name])
    
    summary_data.append({
        'Dataset': dataset_name.upper(),
        'Target_UIDs': n_target,
        'With_Vasprun': n_found,
        'Processed': n_processed,
        'Failed': n_failed
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

print("\n" + "="*70)
print("OUTPUT FILES (Window: 1.0 eV, Descriptor: E_pfrac only)")
print("="*70)
for dataset_name in targets.keys():
    csv_files = saved_csvs.get(dataset_name, [])
    if csv_files:
        for csv_file in csv_files:
            fpath = os.path.join(OUTPUT_DIR, csv_file)
            if os.path.exists(fpath):
                size = os.path.getsize(fpath)
                df_check = pd.read_csv(fpath)
                print(f"\n✓ {csv_file}")
                print(f"    Location: {OUTPUT_DIR}")
                print(f"    Size: {size/1024:.1f} KB")
                print(f"    Rows: {len(df_check)} compounds")
                print(f"    Columns: {list(df_check.columns)}")

print("\n" + "="*70)
print("NOTE: 'unmatched' uses high_order.csv and Inverse-design/unmatched/")
print("      Update dataset CSV if using different unmatched definitions")
print("="*70)


PROCESSING COMPLETE - SUMMARY

    Dataset  Target_UIDs  With_Vasprun  Processed  Failed
     RASHBA           99            99         99       0
DRESSELHAUS           25            25         25       0
  UNMATCHED           78            78         77       1

OUTPUT FILES (Window: 1.0 eV, Descriptor: E_pfrac only)

✓ desc_E_p1.0_rashba.csv
    Location: C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Weight-contribution\contribution-model
    Size: 6.8 KB
    Rows: 99 compounds
    Columns: ['uid', 'Formula', 'alpha_R', 'window', 'E_pfrac_VBM', 'E_pfrac_CBM']

✓ desc_E_p1.0_dresselhaus.csv
    Location: C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Weight-contribution\contribution-model
    Size: 1.7 KB
    Rows: 25 compounds
    Columns: ['uid', 'Formula', 'alpha_R', 'window', 'E_pfrac_VBM', 'E_pfrac_CBM']

✓ desc_E_p1.0_unmatched.csv
    Location: C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Weight-contribution\contribution-model
    Size: 4.9 KB
    Rows: 77 compounds
    Columns: ['uid', 'Formula', 'a

In [49]:
# ============================================================
# VERIFICATION: Compare current E_pfrac vs previous descriptor
# ============================================================
print("\n" + "="*70)
print("VERIFICATION: Compare Current DescE_p1.0 vs Previous DescE_p1.0")
print("="*70)

# Load current output
current_csv = os.path.join(OUTPUT_DIR, "desc_E_p1.0_rashba.csv")
if os.path.exists(current_csv):
    df_current = pd.read_csv(current_csv)
    print(f"\nCURRENT (newly generated):")
    print(f"  File: desc_E_p1.0_rashba.csv")
    print(f"  Shape: {df_current.shape}")
    print(f"  Columns: {list(df_current.columns)}")
    print(f"  E_pfrac_VBM range: [{df_current['E_pfrac_VBM'].min():.4f}, {df_current['E_pfrac_VBM'].max():.4f}]")
    print(f"  E_pfrac_CBM range: [{df_current['E_pfrac_CBM'].min():.4f}, {df_current['E_pfrac_CBM'].max():.4f}]")
    print(f"\n  First 5 rows:")
    print(df_current.head(5).to_string(index=False))
else:
    print(f"⚠ Current CSV not found: {current_csv}")
    df_current = None

# Load previous output (specify path to old desc_E_p1.0_rashba.csv)
# TODO: Update this path to where your previous descriptor CSV is located
PREVIOUS_CSV_PATH = r"C:\Users\AbCMS_Lab\Desktop\Keshav-DDP\Weight-contribution\contribution-model\desc_E_w10.csv"  # Change this to the actual path

if PREVIOUS_CSV_PATH and os.path.exists(PREVIOUS_CSV_PATH):
    df_previous = pd.read_csv(PREVIOUS_CSV_PATH)
    print(f"\n\nPREVIOUS (old descriptor):")
    print(f"  File: {os.path.basename(PREVIOUS_CSV_PATH)}")
    print(f"  Shape: {df_previous.shape}")
    print(f"  Columns: {list(df_previous.columns)}")
    print(f"  E_pfrac_VBM range: [{df_previous['E_pfrac_VBM'].min():.4f}, {df_previous['E_pfrac_VBM'].max():.4f}]")
    print(f"  E_pfrac_CBM range: [{df_previous['E_pfrac_CBM'].min():.4f}, {df_previous['E_pfrac_CBM'].max():.4f}]")
    print(f"\n  First 5 rows:")
    print(df_previous.head(5).to_string(index=False))
    
    # Compare if same shape
    if df_current is not None:
        print(f"\n\nCOMPARISON:")
        if df_current.shape == df_previous.shape:
            print(f"  ✓ Same shape: {df_current.shape}")
            
            # Check if data is identical
            if df_current.equals(df_previous):
                print(f"  ✓ Data is IDENTICAL")
            else:
                print(f"  ✗ Data is DIFFERENT")
                
                # Show differences
                for col in ['E_pfrac_VBM', 'E_pfrac_CBM']:
                    if col in df_current.columns and col in df_previous.columns:
                        diff = (df_current[col] - df_previous[col]).abs()
                        print(f"    {col}: max_diff = {diff.max():.6f}, mean_diff = {diff.mean():.6f}")
        else:
            print(f"  ✗ Different shapes: current {df_current.shape} vs previous {df_previous.shape}")
else:
    if df_current is not None:
        print(f"\n⚠ Previous CSV not found.")
        print(f"  Set PREVIOUS_CSV_PATH to the path of your old descriptor CSV to compare.")


VERIFICATION: Compare Current DescE_p1.0 vs Previous DescE_p1.0

CURRENT (newly generated):
  File: desc_E_p1.0_rashba.csv
  Shape: (99, 6)
  Columns: ['uid', 'Formula', 'alpha_R', 'window', 'E_pfrac_VBM', 'E_pfrac_CBM']
  E_pfrac_VBM range: [0.1341, 0.9468]
  E_pfrac_CBM range: [0.0928, 0.9418]

  First 5 rows:
         uid Formula  alpha_R  window  E_pfrac_VBM  E_pfrac_CBM
001e03f2c095    SSeW    3.288     1.0     0.224009     0.194420
03bcf7dcdaf2  Sn2Te2    4.804     1.0     0.678554     0.835895
04fdd7d1ec5c  ClSbTe    1.018     1.0     0.884091     0.920193
05a06afa3b20 WMo3Se8    1.643     1.0     0.229004     0.177661
0b7696e1f4c9 CrW3Se8    1.756     1.0     0.181304     0.153414


PREVIOUS (old descriptor):
  File: desc_E_w10.csv
  Shape: (99, 10)
  Columns: ['uid', 'Formula', 'alpha_R', 'heavy1_el', 'heavy2_el', 'heavy1_mass', 'heavy2_mass', 'n_elements', 'E_pfrac_VBM', 'E_pfrac_CBM']
  E_pfrac_VBM range: [0.1341, 0.9468]
  E_pfrac_CBM range: [0.0928, 0.9418]

  First 5 row